# 4. Transcribe audio (Speech-to-text)

Currently, Foundry Local supports 3 types of clients: a chat client, an embedding client, and an audio client. We have seen examples for a chat client and an embedding client in previous lessons.

In this lesson, we explore another client, an audio client.

## Models to support audio input

Before starting, list all models which support audio input.

In [1]:
from foundry_local_sdk import Configuration, FoundryLocalManager

# initialize manager and execution providers (see Lesson1)
config = Configuration(app_name="foundry_local_samples")
FoundryLocalManager.initialize(config)
manager = FoundryLocalManager.instance
manager.download_and_register_eps()

# list models which has "audio" in input modalities
models = manager.catalog.list_models()
for m in models:
    if "audio" in m.input_modalities:
        print(f"{m.id} (alias: {m.alias})")

nemotron-speech-streaming-en-0.6b-generic-cpu:3 (alias: nemotron-speech-streaming-en-0.6b)
openai-whisper-base-cuda-gpu:2 (alias: whisper-base)
openai-whisper-large-v3-turbo-cuda-gpu:2 (alias: whisper-large-v3-turbo)
openai-whisper-medium-cuda-gpu:2 (alias: whisper-medium)
openai-whisper-small-cuda-gpu:2 (alias: whisper-small)
openai-whisper-tiny-cuda-gpu:2 (alias: whisper-tiny)


## Transcribe audio (Batch)

In the fisrt example, we use ```whisper-small``` to transcribe a pre-recorded audio file.

In [2]:
model = manager.catalog.get_model("whisper-small")
model.download()
model.load()

Before starting, click [here](https://raw.githubusercontent.com/microsoft/Foundry-Local/refs/heads/main/samples/python/audio-transcription/Recording.mp3) to download the recording file (Recording.mp3) from official GitHub repository, and place it in this working folder.

Now let's transcribe audio (Recording.mp3) using ```whisper-small``` as follows.

In [3]:
# Get audio client
audio_client = model.get_audio_client()

# transcribe audio (Recording.mp3)
result = audio_client.transcribe("Recording.mp3")
print(result.text)

 And lots of times you need to give people more than one link at a time. A band could give their fans a couple of new videos from a live concert, behind the scenes photo gallery, an album to purchase, like these next few links.


In [4]:
model.unload()

## Live Transcription (Realtime)

In the second example, audio input from a microphone is transcribed in real time.

> Note : If you are using remote desktop protocol (RDP), please setup remote audio recording in connection settings.

This example uses PyAudio for capturing microphone inputs.<br>
Before starting, please install ```pyaudio``` in order to run this example as follows.

> Note : Currently, ```pyaudio``` pre-compiled wheel for Python version 3.14 and above is not provided. Use Python versions from 3.8 to 3.13 to install pre-compiled wheels. (Compiling pyaudio is a little bit tricky.)

In [ ]:
!pip install pyaudio

Now download and load the streaming speech model.<br>
Currently, ```nemotron-speech-streaming-en-0.6b``` is the only model to support streaming speech.

In [5]:
model = manager.catalog.get_model("nemotron-speech-streaming-en-0.6b")
model.download()
model.load()

Let's run live transcription as follows.

In this source code :
- Create and start a live transcription session using audio client.
- Retrieve input data from your microphone (using PyAudio) and pass it to above transcription session.
- Transcription results are captured (also by realtime manner) in another thread. (The result is then output by realtime.)

In [ ]:
import threading
import pyaudio
import time

# -----------------------
# create and start a live transcription session
# -----------------------
audio_client = model.get_audio_client()
session = audio_client.create_live_transcription_session()
session.settings.sample_rate = 16000
session.settings.channels = 1
session.settings.language = "en"

session.start()

# -----------------------
# start thread to capture transcription results
# -----------------------
def get_transcription():
    for stream in session.get_stream():
        text = stream.content[0].text if stream.content else ""
        if stream.is_final:
            print(f"\n[Final Text] {text}")
        elif text:
            print(text, end="", flush=True)

live_trans_thread = threading.Thread(
    target=get_transcription,
    daemon=True
)
live_trans_thread.start()

# -----------------------
# retrieve data from microphone and
# pass it to transcription session
# -----------------------
channels = 1
rate = 16000
chunk = 480
p = pyaudio.PyAudio()
stream = p.open(
    format=pyaudio.paInt16,
    channels=channels,
    rate=rate,
    input=True,
    frames_per_buffer=chunk,
)

print("For 10 seconds, speak into your microphone.\n")
end_time = time.time() + 10
while time.time() <= end_time:
    data = stream.read(chunk, exception_on_overflow=False)
    session.append(data)

# -----------------------
# clean-up
# -----------------------
stream.close()
p.terminate()
session.stop()
live_trans_thread.join(timeout=5)

For 10 seconds, speak into your microphone.

 This is a pen
[Final Text]  This is a pen


In [7]:
model.unload()